In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_parquet("../data/processed/df_enriched_final.parquet", engine="fastparquet")

In [3]:
print(df.columns.tolist())

['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'product_name', 'aisle_id', 'department_id', 'aisle', 'department', 'nutriscore_grade', 'main_category', 'health_points', 'unit_price', 'order_value']


In [4]:
df['nutriscore_grade'] = df['nutriscore_grade'].str.upper().str.strip()

nutri_map = {'A':5,'B':4,'C':3,'D':2,'E':1}
df['nutri_score_num'] = df['nutriscore_grade'].map(nutri_map)

df['nutri_missing'] = df['nutri_score_num'].isna().astype(int)

# imputation médiane
df['nutri_score_num'] = df['nutri_score_num'].fillna(df['nutri_score_num'].median())

# healthy flag
df['healthy'] = df['nutri_score_num'].isin([5,4]).astype(int)

In [5]:
mean_price_by_dept = df.groupby('department_id')['unit_price'].mean().rename('dept_mean_price')
df = df.merge(mean_price_by_dept, on='department_id', how='left')

df['relative_price'] = df['unit_price'] / df['dept_mean_price']
df['cheap_product'] = (df['relative_price'] < 1).astype('int8')

In [6]:
order_count = df.groupby('user_id')['order_id'].nunique().rename('total_orders')

avg_days_between_orders = df.groupby('user_id')['days_since_prior_order'].mean().rename('avg_days_between_orders')
recency = df.groupby('user_id')['days_since_prior_order'].last().fillna(0).rename('recency')

reorder_ratio = df.groupby('user_id')['reordered'].mean().rename('reorder_ratio')

basket_sizes = df.groupby(['user_id','order_id']).size().reset_index(name='basket_size')
avg_basket_size = basket_sizes.groupby('user_id')['basket_size'].mean().rename('avg_basket_size')
basket_size_std = basket_sizes.groupby('user_id')['basket_size'].std().fillna(0).rename('basket_size_std')

basket_value = df.groupby(['user_id','order_id'])['unit_price'].sum().reset_index()
avg_basket_value = basket_value.groupby('user_id')['unit_price'].mean().rename('avg_basket_value')

psi = df.groupby('user_id')['cheap_product'].mean().rename('PSI')
price_std_user = df.groupby('user_id')['relative_price'].std().rename('price_std')

HAI = df.groupby('user_id')['healthy'].mean().rename('HAI')
mean_nutri_score = df.groupby('user_id')['nutri_score_num'].mean().rename('mean_nutri_score')

def entropy(series):
    probs = series.value_counts(normalize=True)
    return -(probs*np.log2(probs+1e-9)).sum()

category_entropy = df.groupby('user_id')['department_id'].apply(entropy).rename('category_entropy')

order_frequency = (order_count / (avg_days_between_orders + 1)).rename('order_frequency')

customer_features = pd.concat([
    order_frequency,
    recency,
    reorder_ratio,
    avg_basket_size,
    basket_size_std,
    avg_basket_value,
    psi,
    price_std_user,
    HAI,
    mean_nutri_score,
    category_entropy
], axis=1).reset_index()

In [7]:
total_orders = df['order_id'].nunique()

product_support = (
    df.groupby('product_id')['order_id'].nunique() / total_orders
).rename('support')

total_purchases = df.groupby('product_id').size().rename('total_purchases')
reorder_rate = df.groupby('product_id')['reordered'].mean().rename('reorder_rate')
unique_users = df.groupby('product_id')['user_id'].nunique().rename('unique_users')

avg_price = df.groupby('product_id')['unit_price'].mean().rename('avg_price')
price_std = df.groupby('product_id')['unit_price'].std().fillna(0).rename('price_std')

utility = (avg_price * total_purchases).rename('utility')

product_nutri = df.groupby('product_id')['nutri_score_num'].mean().rename('nutri_score')
healthy_flag = (product_nutri >= 4).astype(int).rename('healthy_flag')

total_users = df['user_id'].nunique()
penetration = (unique_users / total_users).rename('penetration')

aisle_popularity = df.groupby('aisle_id').size().rename('aisle_popularity')
dept_popularity = df.groupby('department_id').size().rename('dept_popularity')

product_features = pd.concat([
    product_support,
    total_purchases,
    reorder_rate,
    unique_users,
    avg_price,
    price_std,
    utility,
    product_nutri,
    healthy_flag,
    penetration
], axis=1).reset_index()

product_features = product_features.merge(
    df[['product_id','aisle_id','department_id']].drop_duplicates(),
    on='product_id', how='left'
)

product_features = product_features.merge(aisle_popularity, on='aisle_id', how='left')
product_features = product_features.merge(dept_popularity, on='department_id', how='left')

print(df.columns.tolist())

df.to_parquet("../data/processed/df_final_for_pipeline.parquet", index=False, engine="fastparquet")

['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'product_name', 'aisle_id', 'department_id', 'aisle', 'department', 'nutriscore_grade', 'main_category', 'health_points', 'unit_price', 'order_value', 'nutri_score_num', 'nutri_missing', 'healthy', 'dept_mean_price', 'relative_price', 'cheap_product']
